In [ ]:
import sys
sys.path.append("..")

from src import config
from src.file_utils import load_file
from src.factors.utils import get_latest_fundamental_period, get_latest_price_date
from src.factors import build_momentum_table, build_quality_table, build_value_table

import pandas as pd

### Load universe

- Use ticker as an index to use `.loc` later

In [3]:
universe = load_file(config.RAW_DATA_DIR / "universe.csv", index_col="Ticker")

In [ ]:
momentum_table = []
quality_table = []
value_table = []

missing_files = []
mismatched_periods = []
missing_dates = []

for ticker in universe.index:
    balance_sheet = load_file(config.FUNDAMENTALS_SAVE_DIR / ticker / "balance_sheet.csv")
    if balance_sheet is None:
        missing_files.append({"ticker": ticker, "missing_file": "balance_sheet"})

    income_statement = load_file(config.FUNDAMENTALS_SAVE_DIR / ticker / "income_statement.csv")
    if income_statement is None:
        missing_files.append({"ticker": ticker, "missing_file": "income_statement"})

    cash_flow = load_file(config.FUNDAMENTALS_SAVE_DIR / ticker / "cash_flow.csv")
    if cash_flow is None:
        missing_files.append({"ticker": ticker, "missing_file": "cash_flow"})

    prices = load_file(config.PRICES_SAVE_DIR / f"{ticker}.csv")
    if prices is None:
        missing_files.append({"ticker": ticker, "missing_file": "prices"})

    metadata = load_file(config.RAW_DATA_DIR / "metadata.csv")
    if metadata is None:
        missing_files.append({"ticker": ticker, "missing_file": "metadata"})

    if any(file is None for file in [balance_sheet, income_statement, cash_flow, prices, metadata]):
        continue

    ticker_country = universe.loc[ticker, "Country"]
    
    for date in config.REBALANCE_DATES:
        latest_balance_sheet_period = get_latest_fundamental_period(balance_sheet, date)
        latest_income_statement_period = get_latest_fundamental_period(income_statement, date)
        latest_cash_flow_period = get_latest_fundamental_period(cash_flow, date)
        latest_price_date = get_latest_price_date(prices, date)

        periods = {
            "latest_balance_sheet_period": latest_balance_sheet_period, 
            "latest_income_statement_period": latest_income_statement_period, 
            "latest_cash_flow_period": latest_cash_flow_period
        }

        if len(set(periods.values())) != 1:
            mismatched_periods.append({
                "ticker": ticker,
                "date": date,
                "balance_sheet_period": latest_balance_sheet_period,
                "income_statement_period": latest_income_statement_period,
                "cash_flow_period": latest_cash_flow_period
            })

        periods_missing = [k for k, v in periods.items() if v is None]
        if len(periods_missing) > 0:
            missing_dates.append({
                "ticker": ticker,
                "date": date,
                "missing_periods": periods_missing,
            })
            continue

        build_quality_table(
            ticker, 
            ticker_country, 
            date, 
            balance_sheet, 
            income_statement, 
            latest_balance_sheet_period, 
            latest_income_statement_period, 
            quality_table
        )
        build_value_table(
            ticker, 
            date, 
            balance_sheet, 
            cash_flow, 
            income_statement, 
            prices,
            metadata,
            latest_balance_sheet_period,
            latest_income_statement_period,
            latest_cash_flow_period,
            latest_price_date,
            value_table
        )